# 2장 1강: 회귀와 분류 모델의 예측 — 실습문제

## 실습 목표

- 금액을 예측하는 회귀와 구매 여부를 예측하는 분류를 구분합니다.
- 분류의 점수, 예측 확률, 최종 클래스가 각각 무엇인지 구분합니다.
- 교안의 함수를 이용해 Sigmoid·Softmax 확률을 계산합니다.
- 임계값 비교와 Argmax로 최종 클래스를 선택합니다.

## 실습 환경 / 데이터

Python과 NumPy를 사용합니다. 외부 데이터 파일은 필요하지 않습니다. 모든 숫자와 계산 방법은 제공된 「2장 1강: 회귀와 분류 모델의 예측」의 본문 예제와 연습문제를 바탕으로 구성했습니다. 주어진 가중치로 출력만 계산하며 모델을 학습시키는 실습은 없습니다.

## 실습 구성과 교안 대응

| 구분 | 내용 | 교안 근거 및 구성 방식 |
|---|---|---|
| 필수 1 / 문제 1-1 | 회귀 예측값과 분류 점수 | 1절 본문의 고객 3명 예제 |
| 필수 1 / 문제 1-2 | Sigmoid와 임계값 | 1절의 점수에 2·3절의 계산 규칙을 연결 |
| 필수 2 / 문제 2-1 | 한 상품의 Softmax와 Argmax | 2·3절 본문의 [2,1,0] 예제 |
| 필수 2 / 문제 2-2 | 고객별 Softmax와 Argmax | 2·3절 본문의 3행 점수 배열 예제 |
| 과제 / 문제 3-1 | 임계값 0.5와 0.7 비교 | 연습문제 3의 소문항 1·2를 나누어 연습 |
| 과제 / 문제 3-2 | 한 상품의 Softmax 계산 복습 | 연습문제 2의 소문항 3에 3절의 Argmax 적용 |

**교안의 연습문제 전체를 그대로 옮긴 자료는 아닙니다.** 교안 본문 예제와 연습문제의 일부를 기초 실습 4문제·복습 과제 2문제로 구성했습니다. 새 공식을 유도하거나 여러 오류 유형을 한꺼번에 분류하는 과제는 요구하지 않습니다.

## 진행 안내

1. NumPy를 불러오는 셀부터 위에서 아래로 실행합니다.
2. 표에서 숫자가 무엇을 뜻하는지 확인한 뒤 계산합니다.
3. 함수 코드는 문제에 제공된 교안의 형태를 그대로 사용해도 됩니다. `return` 앞의 들여쓰기를 유지하세요.
4. 확률을 소수점 아래 세 자리로 표시할 때만 `np.round()`를 씁니다. 임계값과 비교할 때는 반올림 전 확률을 사용합니다.
5. 각 문제의 함수와 변수를 다시 작성합니다. 한 벡터용 Softmax와 여러 행용 Softmax는 사용하는 코드가 다릅니다.
6. 해석 질문의 답은 계산값과 함께 쉬운 문장으로 적으면 됩니다.

## 먼저 읽는 용어와 기호

| 용어·기호 | 쉬운 뜻 |
|---|---|
| 입력 특성 | 계산에 넣는 정보. 여기서는 방문 횟수와 장바구니 추가 횟수 |
| 가중치 w / 편향 b | 입력에 각각 곱할 숫자 / 곱해서 더한 결과에 추가로 더할 숫자 |
| 회귀 | 금액처럼 ‘얼마인가’를 숫자로 예측하는 문제 |
| 분류 / 클래스 | 정해진 종류 중 하나를 고르는 문제 / 그 종류의 이름 |
| 이진 분류 | 두 종류 중 하나를 선택하는 분류. 여기서는 구매·미구매 |
| 다중 클래스 분류 | 세 개 이상의 종류 중 하나를 선택하는 분류. 여기서는 의류·식품·가전 |
| 점수 z / Logit(로짓) | 확률로 바꾸기 전 모델이 계산한 숫자 |
| 확률 p | 모델이 어떤 클래스에 부여한 가능성을 0~1의 형태로 나타낸 값. 0.7은 70% |
| 실제값 y / 예측값 ŷ | 실제로 관측된 정답 / 모델이 계산한 결과. ŷ는 ‘와이 햇’으로 읽음 |
| Sigmoid(시그모이드) | 이진 분류에서 점수 하나를 클래스 1의 확률로 바꾸는 함수 |
| Softmax(소프트맥스) | 한 데이터의 클래스별 점수를 합이 1인 확률들로 바꾸는 함수 |
| Threshold(임계값) | 확률을 0 또는 1로 결정할 때 비교하는 기준값 |
| Argmax(아그맥스) | 가장 큰 값이 있는 위치 번호를 찾는 연산 |
| 인덱스 | 배열의 위치 번호. Python에서는 첫 위치가 0 |
| 행 / 열 / Shape | 가로줄 / 세로줄 / 배열의 모양. (3,2)는 3행 2열 |

이 실습의 이진 분류에서 **0은 미구매, 1은 구매**입니다. 다중 클래스의 인덱스는 **0=의류, 1=식품, 2=가전**입니다. 같은 숫자 1이어도 이진 분류의 클래스 1과 다중 클래스의 인덱스 1은 서로 다른 이름을 가리킵니다.

## 실습 준비

아래 셀을 먼저 실행하세요. NumPy는 숫자 배열을 계산하는 도구이며, `np`는 NumPy를 짧게 부르는 이름입니다.

In [ ]:
import numpy as np

---

## 과제. 필수 실습과 같은 방법으로 복습하기

### 문제 3-1. 같은 확률에 두 임계값 적용하기

#### 문제 설명

교안 연습문제 3의 소문항 1·2를 사용합니다. 이미 구매 확률이 주어져 있으므로 Sigmoid를 다시 적용하지 않습니다.

| 고객 | A | B | C | D |
|---|---:|---:|---:|---:|
| 구매 확률 | 0.85 | 0.40 | 0.65 | 0.20 |

규칙은 **확률이 임계값 이상이면 구매(1), 작으면 미구매(0)**입니다. 같은 확률에 임계값 0.5와 0.7을 각각 적용하세요. 실제 구매 여부는 주어지지 않았습니다.

#### 요구사항

1. `p = np.array([0.85, 0.40, 0.65, 0.20])`를 만드세요.
2. `pred_05 = (p >= 0.5).astype(int)`를 계산하고 출력하세요.
3. `pred_07 = (p >= 0.7).astype(int)`를 계산하고 출력하세요.
4. 답변 작성란에 고객별 두 예측값을 적고, 바뀐 고객을 찾으세요.

#### 작성 도움말

문제 1-2의 비교식을 그대로 사용합니다. 예를 들어 고객 C의 0.65는 0.5 이상이지만 0.7보다 작습니다. 임계값을 바꾸어도 주어진 확률은 그대로입니다.

#### 해석 질문

**Q1.** 임계값 0.5에서 고객 A·B·C·D의 예측은 각각 무엇인가요?

**Q2.** 임계값 0.7에서 고객 A·B·C·D의 예측은 각각 무엇인가요?

**Q3.** 어느 고객의 예측이 바뀌었으며, 그 이유는 무엇인가요?

**Q4.** 임계값을 바꾸면 구매 확률 자체도 바뀌나요? 이 정보만으로 어느 기준이 더 정확한지 알 수 있나요?

#### 제출 결과

- 두 임계값의 예측 배열
- 고객별 비교와 Q1~Q4 답변

In [ ]:
# 문제 3-1. 같은 확률에 두 임계값 적용하기
# 위 데이터와 작성 도움말을 보고 한 단계씩 작성하세요.
# 함수는 문제에 제공된 교안의 코드를 참고하세요.

In [ ]:
p= np.array([0.85, 0.40, 0.65, 0.20])

In [ ]:
pred_05= (p >= 0.5).astype(int)
print("pred_05: ", pred_05)

pred_05:  [1 0 1 0]


In [ ]:
pred_07= (p >= 0.7).astype(int)
print("pred_07: ", pred_07)

pred_07:  [1 0 0 0]


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 
```python
pred_05:  [1 0 1 0]

pred_07:  [1 0 0 0]
```
- C 고객이 구매에서 미구매로 바뀌었음.

**Q1. 임계값 0.5에서 고객 A·B·C·D의 예측은 각각 무엇인가요?**

- 답변: A 구매, B 미구매, C 구매, D 미구매

**Q2. 임계값 0.7에서 고객 A·B·C·D의 예측은 각각 무엇인가요?**

- 답변: A 구매, B 미구매, C 미구매, D 미구매

**Q3. 어느 고객의 예측이 바뀌었으며, 그 이유는 무엇인가요?**

- 답변: C 고객의 예측이 바뀌었다.
- 임계값을 0.7로 설정했기 때문이다.

**Q4. 임계값을 바꾸면 구매 확률 자체도 바뀌나요? 이 정보만으로 어느 기준이 더 정확한지 알 수 있나요?**

- 답변: 구매 확률 자체는 바뀌지 않는다.
- 이 정보만으로는 알 수 없다. 손실함수 값을 계산해서 비교하여야 알 수 있다.

### 문제 3-2. 한 상품의 Softmax와 Argmax 복습하기

#### 문제 설명

교안 연습문제 2의 소문항 3에 있는 점수입니다. 문제 2-1과 같은 방법으로 계산하세요.

| 클래스 | 의류 | 식품 | 가전 |
|---|---:|---:|---:|
| 인덱스 | 0 | 1 | 2 |
| 점수 | 1 | 2 | 1 |

이 한 상품은 세 종류 중 하나로 분류합니다. 교안 3절의 Argmax를 사용해 가장 높은 확률의 클래스도 확인하세요.

#### 요구사항

1. 문제 2-1의 한 벡터용 Softmax 함수를 그대로 작성하세요.
2. `z = np.array([1, 2, 1])`, `classes = np.array(["의류", "식품", "가전"])`를 만드세요.
3. `p = softmax(z)`를 계산하고 `np.round(p, 3)`과 `np.sum(p)`를 출력하세요.
4. `index = np.argmax(p)`를 계산하고 index와 `classes[index]`를 출력하세요.

#### 작성 도움말

한 벡터용 함수에는 axis=1을 쓰지 않습니다. 가장 큰 점수 2를 빼면 [−1,0,−1]이고, 지수값은 약 [0.3679,1,0.3679]입니다. 이 세 값을 더한 뒤 각각을 같은 합으로 나누세요.

#### 해석 질문

**Q1.** 최댓값을 뺀 배열, 지수값, 지수값의 합은 각각 무엇인가요?

**Q2.** 의류·식품·가전의 확률과 확률 합은 각각 얼마인가요?

**Q3.** Argmax 결과와 최종 예측 클래스는 무엇인가요? 인덱스 1은 확률 100%라는 뜻인가요?

**Q4.** 의류와 가전의 확률이 같은 이유는 무엇인가요?

#### 제출 결과

- 한 벡터용 Softmax 함수, 확률·확률 합
- 예측 인덱스·클래스 이름과 Q1~Q4 답변

In [ ]:
# 문제 3-2. 한 상품의 Softmax와 Argmax 복습하기
# 위 데이터와 작성 도움말을 보고 한 단계씩 작성하세요.
# 함수는 문제에 제공된 교안의 코드를 참고하세요.

In [ ]:
def softmax(z):
    exp_z = np.exp(z - np.max(z))
    return exp_z / np.sum(exp_z)

In [ ]:
Z= np.array([1, 2, 1])
classes= np.array(["의류", "식품", "가전"])
print("Z: \n", Z)

Z: 
 [1 2 1]


In [ ]:
p= softmax(Z)
print("np.round(p, 3): \n", np.round(p, 3))
print("np.sum(p): \n", np.sum(p))

np.round(p, 3): 
 [0.212 0.576 0.212]
np.sum(p): 
 1.0


In [ ]:
index= np.argmax(p)
print("index: \n", index)
print("\nclasses[index]: \n", classes[index])

index: 
 1

classes[index]: 
 식품


#### 답변 작성란

**수식 및 계산 과정**

- z = [1, 2, 1]이고 최댓값은 2입니다.
- 최댓값을 뺀 배열: [1−2, 2−2, 1−2] = [−1, 0, −1]
- 지수값: [e^(−1), e^0, e^(−1)] ≈ [0.3679, 1, 0.3679]
- 지수값의 합: 0.3679 + 1 + 0.3679 ≈ 1.7358
- 확률: [0.3679/1.7358, 1/1.7358, 0.3679/1.7358] ≈ [0.212, 0.576, 0.212]

**Q1. 최댓값을 뺀 배열, 지수값, 지수값의 합은 각각 무엇인가요?**

- 답변: 최댓값을 뺀 배열은 [−1, 0, −1], 지수값은 약 [0.3679, 1, 0.3679], 지수값의 합은 약 1.7358입니다.

**Q2. 의류·식품·가전의 확률과 확률 합은 각각 얼마인가요?**

- 답변: 확률의 합은 1이다.

**Q3. Argmax 결과와 최종 예측 클래스는 무엇인가요? 인덱스 1은 확률 100%라는 뜻인가요?**

- 답변: 
```python
index: 
 1

classes[index]: 
 식품
 ```
 - 인덱스 1은 확률 100%를 의미하지 않습니다. 
 - 이유: 

**Q4. 의류와 가전의 확률이 같은 이유는 무엇인가요?**

- 답변: 의류와 가전의 점수가 모두 1로 같기 때문입니다.
- 점수가 같으면 지수값(0.3679)과 확률도 같아집니다. Softmax는 각 점수의 지수값을 같은 합계로 나누기 때문입니다.

---

## 실습 마무리

1. 회귀 예측값, 분류 점수, 예측 확률, 예측 클래스는 어떻게 다른가요?
2. Sigmoid(0)은 얼마이며, 임계값 0.5에서는 어떤 클래스로 정하나요?
3. 구매 확률이 p일 때 미구매 확률은 무엇인가요?
4. Softmax는 무엇을 더한 합으로 각 지수값을 나누나요?
5. 여러 고객의 확률에서 Argmax의 axis=1은 무엇을 뜻하나요?
6. 임계값을 높이거나 가장 높은 확률을 고르면 반드시 더 정확한 예측인가요?

### 실습 마무리 답변 작성란

**1. 회귀 예측값, 분류 점수, 예측 확률, 예측 클래스는 어떻게 다른가요?**

- 답변: 회귀 예측값은 1500원처럼 수치를 예측한 결과
- 분류 점수는 0.8로 확률로 바꾸기 전 숫자를 의미
- 예측 확률은 그 점수에 Sigmoid(이진) / softmax(다중)을 적용한 값을 의미

**2. Sigmoid(0)은 얼마이며, 임계값 0.5에서는 어떤 클래스로 정하나요?**

- 답변: Sigmoid(0) = 0.5이다. 이 값은 임계값 0.5와 같은데, 일반적으로 "확률 ≥ 0.5이면 클래스 1"로 분류하므로 클래스 1로 정한다.

**3. 구매 확률이 p일 때 미구매 확률은 무엇인가요?**

- 답변: 구매와 미구매가 가능한 두 결과 전부이기 때문에 1-p
- A의 구매 확률이 약 0.69이면 미구매 확률은 약 0.31

**4. Softmax는 무엇을 더한 합으로 각 지수값을 나누나요?**

- 답변: 한 데이터의 모든 클래스 점수에 지수 계산을 적용한 값들의 합으로 나눔
- 최대값을 뺀 [0, -1, -2]라면 1, 0.36, 0.13을 더한 1.5를 의미

**5. 여러 고객의 확률에서 Argmax의 axis=1은 무엇을 뜻하나요?**

- 답변: 한 고객을 나타내는 한 행 안에서 클래스별 확률을 비교
- [0.09, 0.66, 0.24]면 두 번째가 확률이 가장 크므로 인덱스 1을 반화(식품)

**6. 임계값을 높이거나 가장 높은 확률을 고르면 반드시 더 정확한 예측인가요?**

- 답변: 보장하지 않음, 임계값은 구매로 판단하는 기준이고 Argmax는 가장 큰 값의 
- 위치를 고르는 규칙, 예측이 맞았는지 여부는 확률값 보다는 실제 비교